# 02 — Kinematics

Inspect raw SAM exports; keep reusable calculations in the Python modules. `src/kinematics.py` contains an exploratory static-offset helper; `src/filtering.py` is a placeholder. Neither provides anatomical angle extraction.


In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path.cwd().resolve()
if not (ROOT / "src" / "comparison_engine.py").is_file():
    ROOT = ROOT.parent
if not (ROOT / "src" / "comparison_engine.py").is_file():
    raise FileNotFoundError("Start Jupyter in markerless_sam/ or notebooks/")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
DATA = ROOT / "lunge_sam_out"
RESULTS = ROOT / "results"
print("Repository:", ROOT, "| Data folder available:", DATA.is_dir())


In [ ]:
SUBJECT, TRIAL, CAMERA = "S11", "lunge_01", "cam0"
RAW = DATA / SUBJECT / TRIAL / CAMERA / "sam_raw.npz"
print("SAM raw NPZ exists:", RAW.exists())

In [ ]:
if RAW.is_file():
    with np.load(RAW, allow_pickle=False) as z:
        print("Available arrays:")
        for key in z.files:
            print(key, z[key].shape, z[key].dtype)
        # Dashboard NPZ exports may contain individual landmark arrays instead
        # of a packed keypoints3d array; timestamps are required in both formats.
        if "time_ms" not in z.files:
            raise ValueError("Raw NPZ must contain time_ms")
        t = np.asarray(z["time_ms"], dtype=float)
        if t.ndim != 1 or len(t) < 2 or not np.all(np.isfinite(t)) or np.any(np.diff(t) <= 0):
            raise ValueError("time_ms must contain at least two finite, increasing timestamps")
        for key in z.files:
            values = z[key]
            if values.ndim == 0 or len(values) != len(t):
                raise ValueError(f"{key} must have one entry per timestamp")
        if "frame" in z.files:
            frames = np.asarray(z["frame"], dtype=float)
            if frames.ndim != 1 or not np.all(np.isfinite(frames)) or np.any(np.diff(frames) <= 0):
                raise ValueError("frame must contain finite, increasing frame indices")
        if "keypoints3d" in z.files:
            points = z["keypoints3d"]
            if points.ndim != 3 or points.shape[-1] != 3:
                raise ValueError("keypoints3d must have shape (frames, landmarks, 3)")
        print("Approximate effective sampling rate:", 1000 / np.median(np.diff(t)), "Hz")
else:
    print("First transfer sam_raw.npz from the inference server.")


In [ ]:
angle_files = sorted(DATA.rglob("*joint_angles.csv"))
print("Processed angle files:", len(angle_files))
for path in angle_files[:10]:
    print(path.relative_to(DATA), list(pd.read_csv(path, nrows=0).columns))


### Implementation next
Transfer the server geometric joint-plane calculations and MHR parent-relative rotation decomposition into tested functions. Standardize hip/knee/ankle sign, coordinate frame and anatomical definitions before comparing to Vicon. Do not select Euler axes using the evaluation trials.